In [1]:
!pip install gitsource

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gitsource]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [40]:
documents[1]

{'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\n```\n\n(You can also use `pip install uv` if you p

Q1. How many lesson pages

In [5]:
len(documents)

72

Q2. Indexing and searching

In [6]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

Q3. RAG

In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

GROQ_MODEL = "llama-3.1-8b-instant"
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [ ]:
def search_docs(query, num_results=5):
    # Retrieve the top matching lesson documents from the minsearch index.
    return index.search(query, num_results=num_results)


def build_context(search_results):
    # Convert retrieved docs into a single context string for the LLM.
    lines = []
    for doc in search_results:
        lines.append(f"Filename: {doc['filename']}")
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    # Build the final user prompt by combining the question and retrieved context.
    context = build_context(search_results)
    return f"""Question: {question}

Context:
{context}
""".strip()


SYSTEM_INSTRUCTIONS = """You are a course teaching assistant.
Answer the QUESTION based only on the CONTEXT.
If the answer is not in the CONTEXT, say: I don't know.
""".strip()

question = "How does the agentic loop keep calling the model until it stops?"
search_results = search_docs(question)
prompt = build_prompt(question, search_results)

In [ ]:
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": prompt},
    ],
)

answer = response.choices[0].message.content
prompt_tokens = response.usage.prompt_tokens

print(answer)
print("Prompt tokens:", prompt_tokens)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01jxwjrv3bec889xfpmd5yftva` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 7206, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
import re
from openai import APIStatusError


def build_context_full(search_results):
    # Build full (untruncated) context to measure the original prompt size.
    lines = []
    for doc in search_results:
        lines.append(f"Filename: {doc['filename']}")
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()


def build_prompt_full(question, search_results):
    # Build a full prompt using all retrieved documents.
    context = build_context_full(search_results)
    return f"""Question: {question}

Context:
{context}
""".strip()


question = "How does the agentic loop keep calling the model until it stops?"
search_results_full = index.search(question, num_results=5)
prompt_full = build_prompt_full(question, search_results_full)

try:
    response_full = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": prompt_full},
        ],
    )
    print("Prompt tokens:", response_full.usage.prompt_tokens)
except APIStatusError as e:
    msg = str(e)
    match = re.search(r"Requested\s+(\d+)", msg)
    if match:
        print("Prompt tokens (full-context request):", int(match.group(1)))
    else:
        print("Could not parse token count from error:")
        print(msg)

Prompt tokens (full-context request): 7206


Q4. Chunking

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [42]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [14]:
len(chunks)

295

Q5. RAG with chunking

In [15]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [16]:
def search_docs(query, num_results=5):
    # Retrieve the top matching lesson documents from the minsearch index.
    return index.search(query, num_results=num_results)


def build_context(search_results):
    # Convert retrieved docs into a single context string for the LLM.
    lines = []
    for doc in search_results:
        lines.append(f"Filename: {doc['filename']}")
        lines.append(doc["content"])
        lines.append("")
    return "\n".join(lines).strip()


def build_prompt(question, search_results):
    # Build the final user prompt by combining the question and retrieved context.
    context = build_context(search_results)
    return f"""Question: {question}

Context:
{context}
""".strip()


SYSTEM_INSTRUCTIONS = """You are a course teaching assistant.
Answer the QUESTION based only on the CONTEXT.
If the answer is not in the CONTEXT, say: I don't know.
""".strip()

question = "How does the agentic loop keep calling the model until it stops?"
search_results = search_docs(question)
prompt = build_prompt(question, search_results)

In [17]:
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": prompt},
    ],
)

answer = response.choices[0].message.content
prompt_tokens = response.usage.prompt_tokens

print(answer)
print("Prompt tokens:", prompt_tokens)

The agentic loop keeps calling the model until it stops by using a `while True` loop with an exit condition. The exit condition is when the model returns a response without any function calls.
Prompt tokens: 2332


Q6. Turning it into an agent


In [34]:
import json
from openai import APIStatusError, BadRequestError


def search_tool_fn(query: str) -> list[dict]:
    """Search the chunk index for relevant lesson content."""
    raw = index.search(query, num_results=5)
    compact = []

    for doc in raw:
        content = doc.get("content", "")
        compact.append({
            "filename": doc.get("filename"),
            "start": doc.get("start"),
            "content": content[:800],
        })

    return compact

In [35]:
search_tool_schema = {
    "type": "function",
    "function": {
        "name": "search_tool_fn",
        "description": "Search lesson chunks relevant to the user question.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query for lesson chunks.",
                }
            },
            "required": ["query"],
        },
    },
}

In [36]:
agent_instructions = (
    "You're a course teaching assistant. "
    "Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)
agent_question = "How does the agentic loop work, and how is it different from plain RAG?"

messages = [
    {"role": "system", "content": agent_instructions},
    {"role": "user", "content": agent_question},
]

search_call_count = 0
final_answer = ""
max_iterations = 6

In [37]:
def run_agent_step(messages):
    response = None

    # Retry transient Groq formatting/rate errors for tool calls.
    for _attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                tools=[search_tool_schema],
                tool_choice="auto",
            )
            break
        except (BadRequestError, APIStatusError):
            continue

    return response

In [38]:
for _ in range(max_iterations):
    response = run_agent_step(messages)

    if response is None:
        final_answer = ""
        break

    assistant_msg = response.choices[0].message
    tool_calls = assistant_msg.tool_calls or []

    messages.append({
        "role": "assistant",
        "content": assistant_msg.content or "",
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in tool_calls
        ],
    })

    if not tool_calls:
        final_answer = assistant_msg.content or ""
        break

    for tc in tool_calls:
        args = json.loads(tc.function.arguments or "{}")

        if tc.function.name == "search_tool_fn":
            search_call_count += 1
            tool_result = search_tool_fn(**args)
        else:
            tool_result = {"error": f"Unknown tool: {tc.function.name}"}

        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tool_result),
        })

In [39]:
print(final_answer)
print("Search tool calls:", search_call_count)
search_call_count

The agentic loop is a fundamental concept in AI agents, where a model is repeatedly asked questions and executes function calls until it produces a final answer with no more tool calls. This loop is the same across various frameworks and is used to send messages, run function calls, and repeat until the model is done.

The difference between the agentic loop and plain RAG (Reinforcement Learning and Architecture Gym) lies in the way the loop is implemented and the tools used. The agentic loop is a more general concept that encompasses various frameworks, including Kestra, OpenAI Agents SDK, and others. Plain RAG, on the other hand, is a specific framework that uses a combination of reinforcement learning and architecture to train agents.

In the agentic loop, the user defines the goal, tools, and optionally a system message, and the framework drives the loop, manages conversation history, and surfaces the result as a task output. In plain RAG, the user defines the problem, the environm

3